In [4]:
import torch
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32 if device == "mps" else torch.bfloat16
print(device)

mps


In [2]:
model = LightOnOcrForConditionalGeneration.from_pretrained("lightonai/LightOnOCR-2-1B", torch_dtype=dtype).to(device)
processor = LightOnOcrProcessor.from_pretrained("lightonai/LightOnOCR-2-1B")

url = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg"

conversation = [{"role": "user", "content": [{"type": "image", "url": url}]}]

inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)
inputs = {k: v.to(device=device, dtype=dtype) if v.is_floating_point() else v.to(device) for k, v in inputs.items()}

output_ids = model.generate(**inputs, max_new_tokens=1024)
generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
output_text = processor.decode(generated_ids, skip_special_tokens=True)
print(output_text)

You are using a model of type mistral3 to instantiate a model of type lighton_ocr. This is not supported for all configurations of models and can yield errors.


Loading weights:   0%|          | 0/532 [00:00<?, ?it/s]

Document No : TD01167104  
Date : 25/12/2018 8:13:39 PM  
Cashier : MANIS  
Member :  

# CASH BILL

<table>
  <thead>
    <tr>
      <th>CODE/DESC</th>
      <th>PRICE</th>
      <th>Disc</th>
      <th>AMOUNT</th>
    </tr>
    <tr>
      <th>QTY</th>
      <th>RM</th>
      <th></th>
      <th>RM</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>9556939040118</td>
      <td>KF MODELLING CLAY KIDDY FISH</td>
      <td></td>
      <td></td>
    </tr>
    <tr>
      <td>1 PC *</td>
      <td>9.000</td>
      <td>0.00</td>
      <td>9.00</td>
    </tr>
    <tr>
      <td colspan="3">Total :</td>
      <td>9.00</td>
    </tr>
    <tr>
      <td colspan="3">Rounding Adjustment :</td>
      <td>0.00</td>
    </tr>
    <tr>
      <td colspan="3">Rounded Total (RM):</td>
      <td>9.00</td>
    </tr>
  </tbody>
</table>


In [ ]:
import requests
import pypdfium2 as pdfium

# --- Configuration ---
PDF_URL = "https://www.arxiv.org/pdf/2512.06641"
MODEL_ID = "lightonai/LightOnOCR-2-1B"
PAGE_SCALE = 2.77  # 200 DPI

# --- Helper Function ---
def get_pdf_page_as_image(url, page_index=0, scale=2.77):
    """Downloads a PDF and renders a specific page as a PIL image."""
    print(f"Downloading PDF from {url}...")
    response = requests.get(url)
    response.raise_for_status()
    
    # Load PDF from bytes
    pdf = pdfium.PdfDocument(response.content)
    
    # Render page
    page = pdf[page_index]
    pil_image = page.render(scale=scale).to_pil()
    return pil_image

# 2. Get Image
image = get_pdf_page_as_image(PDF_URL, page_index=0)

# 3. Prepare Input
# We pass the PIL image object directly into the content list
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image}
        ]
    }
]

# 4. Tokenize
# apply_chat_template handles the image insertion and prompt formatting automatically
inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

# Move inputs to correct device and dtype
inputs = {
    k: v.to(device=device, dtype=dtype) if v.is_floating_point() else v.to(device)
    for k, v in inputs.items()
}

# 5. Generate
print("Generating OCR text...")
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=1024)

# 6. Decode
generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
output_text = processor.decode(generated_ids, skip_special_tokens=True)

print("-" * 40)
print(output_text)
print("-" * 40)

# its slow, took 9m to process 1 page.

Generating OCR text...
----------------------------------------
arXiv:2512.06641v1 [cs.IR] 7 Dec 2025

# An Index-based Approach for Efficient and Effective Web Content Extraction

Yihan Chen  
University of Science and Technology of China  
Hefei, China  
chenyihan@mail.ustc.edu.cn

Benfeng Xu  
University of Science and Technology of China  
Hefei, China  
benfeng@mail.ustc.edu.cn

Xiaorui Wang  
Metastone Technology  
Beijing, China

Zhendong Mao  
University of Science and Technology of China  
Hefei, China

## Abstract

As web agents (e.g., Deep Research) routinely consume massive volumes of web pages to gather and analyze information, LLM context management—under large token budgets and low signal density—emerges as a foundational, high-importance, and technically challenging problem for agentic and RAG pipelines. Existing solutions for extracting relevant content are inadequate: generative extraction models suffer from high latency, rule-based heuristics lack adaptability, and c

In [8]:
# Using vLLM to see if inference is sped up
# run: vllm serve lightonai/LightOnOCR-2-1B \
#    --limit-mm-per-prompt '{"image": 1}' --mm-processor-cache-gb 0 --no-enable-prefix-caching
import base64
import requests
import pypdfium2 as pdfium
import io

ENDPOINT = "http://localhost:8000/v1/chat/completions"
MODEL = "lightonai/LightOnOCR-2-1B"

# Download PDF from arXiv
pdf_url = "https://www.arxiv.org/pdf/2512.06641"
pdf_data = requests.get(pdf_url).content

# Open PDF and convert first page to image
pdf = pdfium.PdfDocument(pdf_data)
page = pdf[0]
# Render at 200 DPI (scale factor = 200/72 ≈ 2.77)
pil_image = page.render(scale=2.77).to_pil()

# Convert to base64
buffer = io.BytesIO()
pil_image.save(buffer, format="PNG")
image_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')

# Make request
payload = {
    "model": MODEL,
    "messages": [{
        "role": "user",
        "content": [{
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{image_base64}"}
        }]
    }],
    "max_tokens": 4096,
    "temperature": 0.2,
    "top_p": 0.9,
}

response = requests.post(ENDPOINT, json=payload)
text = response.json()['choices'][0]['message']['content']
print(text)


arXiv:2512.06641v1 [cs.IR] 7 Dec 2025

# An Index-based Approach for Efficient and Effective Web Content Extraction

Yihan Chen  
University of Science and Technology of China  
Hefei, China  
chenyihan@mail.ustc.edu.cn

Benfeng Xu  
University of Science and Technology of China  
Hefei, China  
benfeng@mail.ustc.edu.cn

Xiaorui Wang  
Metastone Technology  
Beijing, China

Zhendong Mao  
University of Science and Technology of China  
Hefei, China

## Abstract

As web agents (e.g., Deep Research) routinely consume massive volumes of web pages to gather and analyze information, LLM context management—under large token budgets and low signal density—emerges as a foundational, high-importance, and technically challenging problem for agentic and RAG pipelines. Existing solutions for extracting relevant content are inadequate: generative extraction models suffer from high latency, rule-based heuristics lack adaptability, and chunk-and-rerank methods are blind to webpage structure. To overc

In [ ]:
## MLX implementation: DOES NOT WORK

import requests
import pypdfium2 as pdfium
import io
from mlx_vlm import load, generate
from mlx_vlm.prompt_utils import apply_chat_template

# --- Configuration ---
# Use the MLX-converted version. 
# Options: "mlx-community/LightOnOCR-2-1B-bf16" (Best Quality) 
# or "mlx-community/LightOnOCR-2-1B-4bit" (Fastest, lower RAM)
MODEL_PATH = "mlx-community/LightOnOCR-2-1B-bf16"

# 1. Load Model (Native Metal)
print(f"Loading {MODEL_PATH} into Metal...")
model, processor = load(MODEL_PATH)
config = model.config

# 2. Prepare Image (Same as your code)
pdf_url = "https://www.arxiv.org/pdf/2512.06641"
print(f"Downloading {pdf_url}...")
pdf_data = requests.get(pdf_url).content

pdf = pdfium.PdfDocument(pdf_data)
page = pdf[0]
# LightOnOCR recommends high DPI (e.g., 200 DPI -> scale ~2.77)
pil_image = page.render(scale=2.77).to_pil()

# 3. Prepare Prompt
# mlx_vlm handles the chat template for us
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            # LightOnOCR requires a text prompt to trigger the extraction behavior.
            # This is the official prompt for the v2 model:
            {"type": "text", "text": "Extract all text from this page in natural reading order, using Markdown for tables and LaTeX for equations."}
        ]
    }
]

prompt = apply_chat_template(
    processor, 
    config, 
    conversation, 
    add_generation_prompt=True
)

# 4. Run Inference
# This runs completely locally on your GPU/NPU
print("Running inference...")
output = generate(
    model, 
    processor, 
    image=[pil_image], 
    prompt=prompt,
    max_tokens=4096,
    temperature=0.2,
    verbose=True # Prints tokens as they generate
)

print("-" * 40)
print(output)
print("-" * 40)

Loading mlx-community/LightOnOCR-2-1B-bf16 into Metal...


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Running inference...
Files: [<PIL.Image.Image image mode=RGB size=1696x2194 at 0x12DF92F10>] 

Prompt: <|im_start|>system<|im_end|>
<|im_start|>user
Extract all text from this page in natural reading order, using Markdown for tables and LaTeX for equations.
<|im_end|>
<|im_start|>assistant

Extract all text from this page in natural reading order, using Markdown for tables and LaTeX for equations.

<table>
  <tr>
    <td>1</td>
    <td>2</td>
    <td>3</td>
  </tr>
  <tr>
    <td>4</td>
    <td>5</td>
    <td>6</td>
  </tr>
</table>

$$
\frac{1}{2} \times \frac{3}{4} = \frac{3}{8}
$$

- Item 1
- Item 2
- Item 3

1. First item
2. Second item
3. Third item

- Item 1
- Item 2
- Item 3

- Item 1
- Item 2
- Item 3

- Item 1
- Item 2
- Item 3

- Item 1
- Item 2
- Item 3

- Item 1
- Item 2
- Item 3

- Item 1
- Item 2
- Item 3

- Item 1
- Item 2
- Item 3

- Item 1
- Item 2
- Item 3

- Item 1
- Item 2
- Item 3

- Item 1
- Item 2
- Item 3

- Item 1
- Item 2
- Item 3

- Item 1
- Item 2
- Item 3



In [ ]:
import requests
import pypdfium2 as pdfium
from mlx_vlm import load, generate

# 1. Load Model
# Using the bf16 variant as requested
model_path = "mlx-community/LightOnOCR-2-1B-bf16"
model, processor = load(model_path)

# 2. Download and Render PDF
pdf_url = "https://www.arxiv.org/pdf/2512.06641"
print(f"Downloading {pdf_url}...")
pdf_data = requests.get(pdf_url).content

# Initialize PDF document
pdf = pdfium.PdfDocument(pdf_data)

# Select the first page (Index 0)
page = pdf[0]

# LightOnOCR recommends high DPI. 
# scale=2.77 corresponds roughly to 200 DPI for standard documents.
pil_image = page.render(scale=2.77).to_pil()

# 3. Generate OCR Output
# Note: For strict OCR, a prompt like "Convert to markdown." or just "OCR" 
# often works better than "Describe this image." for this specific model.
output = generate(
    model,
    processor,
    image=[pil_image],
    prompt="Convert the text in this image to Markdown.",
    max_tokens=4096,  # Increased tokens for dense academic papers
    temp=0.0,
    verbose=True
)

print("\n--- OCR Result ---")
print(output)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Files: [<PIL.Image.Image image mode=RGB size=1696x2194 at 0x12DABE6D0>] 

Prompt: Convert the text in this image to Markdown.
 The text is formatted as a simple list of items, with no tables, equations, or complex structures. The Markdown output is straightforward and easy to read.

The text in the image is as follows:

1. A simple list of items.
2. No tables, equations, or complex structures.

The Markdown output is as follows:

1. A simple list of items.
2. No tables, equations, or complex structures.

The output is as follows:

1. A simple list of items.
2. No tables, equations, or complex structures.

This is the final output in Markdown format.
Prompt: 9 tokens, 12.525 tokens-per-sec
Generation: 120 tokens, 166.661 tokens-per-sec
Peak memory: 6.034 GB

--- OCR Result ---
GenerationResult(text=' The text is formatted as a simple list of items, with no tables, equations, or complex structures. The Markdown output is straightforward and easy to read.\n\nThe text in the image is as fo

: 